# Long-Hair Gender Identification - Kaggle Training

Attach a UTKFace dataset, enable GPU, run all cells, then download `/kaggle/working/long_hair_gender_models.zip`.

In [ ]:
import os, re, json, random, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

SEED = 42
EPOCHS_AGE = 8
EPOCHS_HAIR = 6
EPOCHS_GENDER = 6
BATCH_SIZE = 32
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

def find_utkface_root():
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        jpgs = [f for f in files[:200] if re.match(r'^\d+_[01]_\d+_.+\.jpg', f)]
        if jpgs:
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Could not find UTKFace files under /kaggle/input')
    return candidates[0]

DATA_DIR = find_utkface_root()
print('Using UTKFace dir:', DATA_DIR)

rows = []
for p in Path(DATA_DIR).glob('*.jpg*'):
    m = re.match(r'^(\d+)_([01])_\d+_.+', p.name)
    if not m:
        continue
    age = int(m.group(1)); gender_id = int(m.group(2))
    if 1 <= age <= 100:
        rows.append({'path': str(p), 'age': age, 'gender': 'Female' if gender_id == 1 else 'Male'})
df = pd.DataFrame(rows)
df['target_age'] = ((df.age >= 20) & (df.age <= 30)).astype(int)
df['strat'] = df['target_age'].astype(str) + '_' + df['gender']
df = df[df.groupby('strat').strat.transform('count') >= 3].reset_index(drop=True)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df.strat, random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df.strat, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

def decode_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    return tf.keras.applications.mobilenet_v2.preprocess_input(image)

def make_ds(frame, labels, batch=BATCH_SIZE, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, labels.astype('float32')))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    ds = ds.map(lambda p, y: (decode_image(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def build_age_model():
    base = MobileNetV2(include_top=False, weights='imagenet', input_shape=(224,224,3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='relu')(x)
    model = keras.Model(base.input, out, name='age_estimator')
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=keras.losses.Huber(), metrics=['mae'])
    return model

def build_binary_model(name):
    base = MobileNetV2(include_top=False, weights='imagenet', input_shape=(224,224,3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(base.input, out, name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

callbacks = [keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
age_model = build_age_model()
age_model.fit(make_ds(train_df, train_df.age.values), validation_data=make_ds(val_df, val_df.age.values, shuffle=False), epochs=EPOCHS_AGE, callbacks=callbacks)
age_model.save(MODEL_DIR / 'age_estimator.keras')

target_train = train_df[(train_df.age >= 20) & (train_df.age <= 30)]
target_val = val_df[(val_df.age >= 20) & (val_df.age <= 30)]
hair_model = build_binary_model('hair_length_classifier')
hair_labels_train = (target_train.gender == 'Female').astype('float32').values
hair_labels_val = (target_val.gender == 'Female').astype('float32').values
hair_model.fit(make_ds(target_train, hair_labels_train), validation_data=make_ds(target_val, hair_labels_val, shuffle=False), epochs=EPOCHS_HAIR, callbacks=callbacks)
hair_model.save(MODEL_DIR / 'hair_classifier.keras')

outside_train = train_df[(train_df.age < 20) | (train_df.age > 30)]
outside_val = val_df[(val_df.age < 20) | (val_df.age > 30)]
gender_model = build_binary_model('gender_predictor')
gender_labels_train = (outside_train.gender == 'Male').astype('float32').values
gender_labels_val = (outside_val.gender == 'Male').astype('float32').values
gender_model.fit(make_ds(outside_train, gender_labels_train), validation_data=make_ds(outside_val, gender_labels_val, shuffle=False), epochs=EPOCHS_GENDER, callbacks=callbacks)
gender_model.save(MODEL_DIR / 'gender_predictor.keras')

config = {'seed': SEED, 'model_architecture': 'MobileNetV2', 'input_shape': [224,224,3], 'notes': 'Gender model label mapping: sigmoid >= 0.5 means Male. Hair model sigmoid >= 0.5 means long hair.'}
(MODEL_DIR / 'config.json').write_text(json.dumps(config, indent=2))
zip_path = '/kaggle/working/long_hair_gender_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.iterdir():
        z.write(p, arcname=p.name)
print('Download:', zip_path)
print(sorted([p.name for p in MODEL_DIR.iterdir()]))